# Notebook 03：因子构造

本 Notebook 使用月度研究面板和日频行情计算 README 中定义的价值、质量、动量、低波动、流动性和市值因子。原始因子按月末 `(date, stock_code)` 对齐；随后在每个调仓日进行 1%/99% 去极值、方向统一和横截面 Z-score 标准化。

## 1. 环境与路径

In [ ]:
from pathlib import Path
import gc
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.factors import (
    CORE_FACTOR_COLUMNS,
    FACTOR_PANEL_CONTEXT_COLUMNS,
    OPTIONAL_FACTOR_COLUMNS,
    build_raw_factor_panel,
    build_zscore_factor_panel,
    save_factor_panels,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MONTHLY_PANEL_PATH = PROCESSED_DIR / "monthly_panel.parquet"
PRICE_DAILY_PATH = PROCESSED_DIR / "price_daily.parquet"
FACTOR_COLUMNS = (*CORE_FACTOR_COLUMNS, *OPTIONAL_FACTOR_COLUMNS)
OUTPUT_COLUMNS = (*FACTOR_PANEL_CONTEXT_COLUMNS, *FACTOR_COLUMNS)

print(f"项目目录：{PROJECT_ROOT}")
print(f"保留的上下文字段：{FACTOR_PANEL_CONTEXT_COLUMNS}")
print(f"因子列：{FACTOR_COLUMNS}")

## 2. 读取月度面板与日频行情

财务类因子直接使用月度面板。动量、低波动和 Amihud 因子需要日频历史；这里只读取计算所需字段，以减少内存占用。

In [ ]:
monthly_panel = pd.read_parquet(MONTHLY_PANEL_PATH)
price_daily = pd.read_parquet(
    PRICE_DAILY_PATH,
    columns=["date", "stock_code", "close", "adj_factor", "amount"],
)

assert not monthly_panel.duplicated(["date", "stock_code"]).any()
assert not price_daily.duplicated(["date", "stock_code"]).any()

print(
    f"月度面板：{monthly_panel.shape}，"
    f"区间 {monthly_panel['date'].min():%Y-%m-%d} 至 {monthly_panel['date'].max():%Y-%m-%d}"
)
print(
    f"日频行情：{price_daily.shape}，"
    f"区间 {price_daily['date'].min():%Y-%m-%d} 至 {price_daily['date'].max():%Y-%m-%d}"
)

## 3. 构造原始因子面板

行情因子的回看期按全市场交易日计算，并使用复权收盘价。精确回看日缺少行情时保留缺失值，不跨越停牌期取股票自身的上一条记录。输出面板只保留主键、标签、股票池状态、行业/市值控制变量和因子，不重复保存行情、财务原始值及中间清洗字段。

In [ ]:
factor_panel_raw = build_raw_factor_panel(
    monthly_panel,
    price_daily,
    include_optional=True,
)

assert len(factor_panel_raw) == len(monthly_panel)
assert not factor_panel_raw.duplicated(["date", "stock_code"]).any()
assert tuple(factor_panel_raw.columns) == OUTPUT_COLUMNS

del price_daily
gc.collect()

raw_quality = pd.DataFrame({
    "非缺失数量": factor_panel_raw[list(FACTOR_COLUMNS)].notna().sum(),
    "缺失率": factor_panel_raw[list(FACTOR_COLUMNS)].isna().mean(),
    "无穷值数量": np.isinf(factor_panel_raw[list(FACTOR_COLUMNS)]).sum(),
})
raw_quality

## 4. 横截面去极值与标准化

每个调仓日独立执行 1%/99% Winsorization 和 Z-score。LOWVOL 在原始定义中已经取负号；Amihud 在标准化时反向，使标准化因子统一为数值越大越好。Size 作为控制变量保留原方向。

In [ ]:
factor_panel_zscore = build_zscore_factor_panel(factor_panel_raw)

zscore_summary = factor_panel_zscore.groupby("date")[list(FACTOR_COLUMNS)].agg(
    ["mean", lambda values: values.std(ddof=0)]
)
zscore_summary.columns = [
    f"{factor}_{stat if stat == 'mean' else 'std'}"
    for factor, stat in zscore_summary.columns
]
print(
    "各月因子均值绝对值最大值：",
    zscore_summary.filter(like="_mean").abs().max().max(),
)
factor_panel_zscore[["date", "stock_code", *FACTOR_COLUMNS]].head()

## 5. 保存结果

In [ ]:
raw_path, zscore_path = save_factor_panels(
    factor_panel_raw,
    factor_panel_zscore,
    output_directory=PROCESSED_DIR,
)

print(f"原始因子面板：{raw_path}，{raw_path.stat().st_size / 1024**2:.1f} MB")
print(f"标准化因子面板：{zscore_path}，{zscore_path.stat().st_size / 1024**2:.1f} MB")

## 6. 回读校验

回读两份输出的主键和因子字段，确认写入结果完整。

In [ ]:
saved_raw = pd.read_parquet(raw_path)
saved_zscore = pd.read_parquet(zscore_path)

assert tuple(saved_raw.columns) == OUTPUT_COLUMNS
assert tuple(saved_zscore.columns) == OUTPUT_COLUMNS
assert saved_raw.shape == saved_zscore.shape
assert len(saved_raw) == len(monthly_panel)
assert saved_raw[["date", "stock_code"]].equals(
    saved_zscore[["date", "stock_code"]]
)
assert not saved_raw.duplicated(["date", "stock_code"]).any()
assert not saved_zscore.duplicated(["date", "stock_code"]).any()

print(
    f"校验完成：两份精简因子面板均包含 {len(saved_raw):,} 行、"
    f"{len(FACTOR_PANEL_CONTEXT_COLUMNS)} 个上下文字段和 {len(FACTOR_COLUMNS)} 个因子。"
)